In [1]:
"""
MAXIM (HF SavedModel) + Trainable Adapter Head (TF 2.15.x)
---------------------------------------------------------
This avoids Keras graph integration issues (KeyError: '') by NOT embedding the SavedModel
inside a Functional model + model.fit(). Instead it uses a custom training loop.

Folder structure:
data/
  train/
    input/   (hazy / rainy / lowlight images)
    target/  (clean images)
  val/
    input/
    target/

Change REPO_ID for task:
- Dehaze outdoor: google/maxim-s2-dehazing-sots-outdoor
- Dehaze indoor : google/maxim-s2-dehazing-sots-indoor
- Derain streaks: google/maxim-s2-deraining-rain13k
- Derain drops  : google/maxim-s2-deraining-raindrop
- Low-light     : google/maxim-s2-enhancement-lol
"""

"\nMAXIM (HF SavedModel) + Trainable Adapter Head (TF 2.15.x)\n---------------------------------------------------------\nThis avoids Keras graph integration issues (KeyError: '') by NOT embedding the SavedModel\ninside a Functional model + model.fit(). Instead it uses a custom training loop.\n\nFolder structure:\ndata/\n  train/\n    input/   (hazy / rainy / lowlight images)\n    target/  (clean images)\n  val/\n    input/\n    target/\n\nChange REPO_ID for task:\n- Dehaze outdoor: google/maxim-s2-dehazing-sots-outdoor\n- Dehaze indoor : google/maxim-s2-dehazing-sots-indoor\n- Derain streaks: google/maxim-s2-deraining-rain13k\n- Derain drops  : google/maxim-s2-deraining-raindrop\n- Low-light     : google/maxim-s2-enhancement-lol\n"

In [2]:
import os
from glob import glob
import numpy as np
from PIL import Image

import tensorflow as tf
from huggingface_hub import hf_hub_download


In [3]:
# ============================================================
# 0) CONFIG
# ============================================================

REPO_ID = "google/maxim-s2-dehazing-sots-outdoor"   # <-- change if needed
DATA_ROOT = r".\data"                              # <-- change if needed

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-4

SAVEDMODEL_DIR = os.path.abspath("./maxim_savedmodel")
ADAPTER_BEST_WEIGHTS = os.path.abspath("./adapter_best.weights.h5")

AUTOTUNE = tf.data.AUTOTUNE
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff")



In [4]:
# ============================================================
# 1) Download SavedModel from Hugging Face
# ============================================================

def download_savedmodel(repo_id: str, out_dir: str):
    os.makedirs(os.path.join(out_dir, "variables"), exist_ok=True)
    files = [
        "saved_model.pb",
        "keras_metadata.pb",
        "variables/variables.index",
        "variables/variables.data-00000-of-00001",
    ]
    for f in files:
        hf_hub_download(
            repo_id=repo_id,
            filename=f,
            local_dir=out_dir,
            local_dir_use_symlinks=False,
        )
    return out_dir

print("Downloading:", REPO_ID)
download_savedmodel(REPO_ID, SAVEDMODEL_DIR)

print("Loading SavedModel backbone...")
base_model = tf.keras.models.load_model(SAVEDMODEL_DIR, compile=False)
base_model.trainable = False  # freeze backbone



Downloading: google/maxim-s2-dehazing-sots-outdoor


C:\Priya\Python\envs\maxim_tf215\lib\site-packages\huggingface_hub\utils\_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading SavedModel backbone...




In [5]:
# ============================================================
# 2) Utilities to safely select a usable output tensor
# ============================================================

def flatten_tensors(y):
    out = []
    if isinstance(y, tf.Tensor):
        out.append(y)
    elif isinstance(y, (list, tuple)):
        for v in y:
            out.extend(flatten_tensors(v))
    elif isinstance(y, dict):
        for v in y.values():
            out.extend(flatten_tensors(v))
    return out

def select_best_output(y, prefer_hw=None):
    """
    y: nested outputs (tensor/list/dict)
    prefer_hw: (H, W) tuple (optional). If provided, prefer outputs matching this size.
    returns: a 4D tensor [B,H,W,C]
    """
    tensors = [t for t in flatten_tensors(y) if isinstance(t, tf.Tensor) and len(t.shape) == 4]
    if not tensors:
        raise ValueError("No 4D tensors found in backbone outputs.")

    if prefer_hw is not None:
        H, W = prefer_hw
        same = [t for t in tensors if (t.shape[1] == H and t.shape[2] == W)]
        if same:
            return same[-1]  # often final prediction at that resolution

    # fallback: max spatial area
    def area(t):
        h = t.shape[1] if t.shape[1] is not None else 0
        w = t.shape[2] if t.shape[2] is not None else 0
        return int(h) * int(w)

    return max(tensors, key=area)


# Backbone predict: keep eager-friendly and stable; wrap in tf.function for speed.
@tf.function
def backbone_predict(x):
    """
    x: [B, IMG_SIZE, IMG_SIZE, 3] float32 in [0,1]
    returns: [B, IMG_SIZE, IMG_SIZE, 3] float32 in [0,1] (upsampled if needed)
    """
    y = base_model(x, training=False)
    pred = select_best_output(y, prefer_hw=(IMG_SIZE, IMG_SIZE))

    # Some checkpoints still emit smaller outputs; ensure correct size
    pred = tf.image.resize(pred, (IMG_SIZE, IMG_SIZE), method="bilinear")
    pred = tf.clip_by_value(pred, 0.0, 1.0)
    return pred


In [6]:
# Quick sanity check
_tmp = backbone_predict(tf.random.uniform([1, IMG_SIZE, IMG_SIZE, 3]))
print("Backbone prediction shape:", _tmp.shape)


Backbone prediction shape: (1, 256, 256, 3)


In [7]:
# ============================================================
# 3) Adapter head (trainable residual refinement)
# ============================================================

adapter = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(3, 3, padding="same", activation=None),
], name="adapter_head")

optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

def restoration_loss(y_true, y_pred):
    # Both are [B, IMG_SIZE, IMG_SIZE, 3] in [0,1]
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = tf.reduce_mean(1.0 - tf.image.ssim(y_true, y_pred, max_val=1.0))
    return l1 + 0.2 * ssim



In [8]:
# ============================================================
# 4) Dataset: pair by stem (supports mixed extensions)
# ============================================================

def list_images(folder):
    paths = []
    for ext in IMG_EXTS:
        paths.extend(glob(os.path.join(folder, f"*{ext}")))
        paths.extend(glob(os.path.join(folder, f"*{ext.upper()}")))
    return sorted(paths)

def build_pairs(input_dir, target_dir):
    in_paths = list_images(input_dir)
    tg_paths = list_images(target_dir)

    tg_map = {os.path.splitext(os.path.basename(p))[0]: p for p in tg_paths}

    pairs, missing = [], []
    for p in in_paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        if stem in tg_map:
            pairs.append((p, tg_map[stem]))
        else:
            missing.append(stem)

    print(f"[PAIRING] {os.path.basename(os.path.dirname(input_dir))} "
          f"Inputs: {len(in_paths)} | Targets: {len(tg_paths)} | Paired: {len(pairs)}")
    if missing:
        print("[PAIRING] Missing targets (first 10):", missing[:10])

    if not pairs:
        raise ValueError(f"No pairs found. Check {input_dir} and {target_dir}.")
    return pairs

def _load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)  # [0,1]
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])  # critical for batching
    return img

def _augment(x, y):
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x)
        y = tf.image.flip_left_right(y)
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_up_down(x)
        y = tf.image.flip_up_down(y)
    return x, y

def make_dataset(split):
    in_dir = os.path.join(DATA_ROOT, split, "input")
    tg_dir = os.path.join(DATA_ROOT, split, "target")

    pairs = build_pairs(in_dir, tg_dir)
    in_list, tg_list = zip(*pairs)

    ds = tf.data.Dataset.from_tensor_slices((list(in_list), list(tg_list)))

    def _map(in_p, tg_p):
        return _load_image(in_p), _load_image(tg_p)

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE).shuffle(512, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset("train")
val_ds = make_dataset("val")


[PAIRING] train Inputs: 984 | Targets: 984 | Paired: 984
[PAIRING] val Inputs: 984 | Targets: 984 | Paired: 984


In [9]:
# Sanity check shapes
xb, yb = next(iter(train_ds))
bp = backbone_predict(xb)
print("Sanity - xb:", xb.shape, "yb:", yb.shape, "backbone_pred:", bp.shape)


Sanity - xb: (8, 256, 256, 3) yb: (8, 256, 256, 3) backbone_pred: (8, 256, 256, 3)


In [10]:
# ============================================================
# 5) Training: custom loop (no model.fit)
# ============================================================

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        base_pred = backbone_predict(x)                 # frozen
        delta = adapter(base_pred, training=True)       # trainable
        out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)
        loss = restoration_loss(y, out)

    grads = tape.gradient(loss, adapter.trainable_variables)

    # Optional: guard against None grads
    grads_and_vars = [(g, v) for g, v in zip(grads, adapter.trainable_variables) if g is not None]
    optimizer.apply_gradients(grads_and_vars)
    return loss

@tf.function
def val_step(x, y):
    base_pred = backbone_predict(x)
    delta = adapter(base_pred, training=False)
    out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)
    loss = restoration_loss(y, out)
    return loss

best_val = float("inf")

print("\nTraining adapter head (custom loop)...")
for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    train_losses = []
    for x, y in train_ds:
        loss = train_step(x, y)
        train_losses.append(loss)
    train_loss = tf.reduce_mean(train_losses).numpy().item()

    # ---- validate ----
    val_losses = []
    for x, y in val_ds:
        vloss = val_step(x, y)
        val_losses.append(vloss)
    val_loss = tf.reduce_mean(val_losses).numpy().item()

    print(f"Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

    # ---- checkpoint best adapter ----
    if val_loss < best_val:
        best_val = val_loss
        adapter.save_weights(ADAPTER_BEST_WEIGHTS)
        print("  ✔ Saved best adapter weights:", ADAPTER_BEST_WEIGHTS)

print("Done. Best val_loss:", best_val)




Training adapter head (custom loop)...
Epoch 1/5 - train_loss: 0.0373 - val_loss: 0.0282
  ✔ Saved best adapter weights: C:\Users\KIIT\Documents\Dehaze\adapter_best.weights.h5
Epoch 2/5 - train_loss: 0.0270 - val_loss: 0.0266
  ✔ Saved best adapter weights: C:\Users\KIIT\Documents\Dehaze\adapter_best.weights.h5
Epoch 3/5 - train_loss: 0.0266 - val_loss: 0.0265
  ✔ Saved best adapter weights: C:\Users\KIIT\Documents\Dehaze\adapter_best.weights.h5
Epoch 4/5 - train_loss: 0.0264 - val_loss: 0.0261
  ✔ Saved best adapter weights: C:\Users\KIIT\Documents\Dehaze\adapter_best.weights.h5
Epoch 5/5 - train_loss: 0.0261 - val_loss: 0.0258
  ✔ Saved best adapter weights: C:\Users\KIIT\Documents\Dehaze\adapter_best.weights.h5
Done. Best val_loss: 0.025795528665184975


In [6]:
# ============================================================
# 6) Inference: backbone + adapter
# ============================================================

def enhance_image(input_path, output_path, weights_path=".\adapter_best"):
    adapter.load_weights(weights_path)

    img = Image.open(input_path).convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    x = tf.convert_to_tensor(arr)[None, ...]
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))

    base_pred = backbone_predict(x)
    delta = adapter(base_pred, training=False)
    out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)[0].numpy()

    Image.fromarray((out * 255).astype(np.uint8)).save(output_path)
    print("Wrote:", output_path)



In [7]:
# Example:
enhance_image(r"./data/val/input/0010.jpg", r"./0010_output.jpg")


NameError: name 'adapter' is not defined